In [ ]:
%pylab inline
import scanpy as sc
import pandas as pd

In [ ]:
def postprocess(adata,view=True):
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable]
    print('after select HVG',adata.shape)
    sc.pp.scale(adata)
    sc.tl.pca(adata)
    sc.pp.neighbors(adata)
    if view:
        sc.tl.tsne(adata)
        sc.tl.umap(adata)
    return adata

In [ ]:
def cluster_k_leiden(embadata,n_cluster):
    max_steps=120
    this_step = 0
    this_min = 0
    this_max = 2
    print('reference cluster number',n_cluster)
    while this_step < max_steps:
        this_resolution = this_min + ((this_max-this_min)/2)
        sc.tl.leiden(embadata,resolution=this_resolution,random_state=42)
        this_clusters = embadata.obs['leiden'].nunique()
        if this_clusters > n_cluster:
            this_max = this_resolution
        elif this_clusters < n_cluster:
            this_min = this_resolution
        else:break
        this_step+=1
    if this_step==max_steps:
        print('Cannot find the number of clusters')
        print('Use resolution',this_resolution)
    else:
        print('use resolution',this_resolution)
     # leiden
    sc.tl.leiden(embadata,resolution=this_resolution,random_state=42,key_added=f'cluster_{n_cluster}')

In [ ]:
refdf = pd.read_csv('./data/baron_human_ref_19264_fromsaver.csv',index_col=0)
refAdata = sc.AnnData(refdf)
refAdata = postprocess(refAdata,True)

In [ ]:
sc.tl.leiden(refAdata,resolution=0.6)

In [ ]:
numcls = refAdata.obs.leiden.unique().shape[0]
numcls

In [ ]:
numcls=7
imputeemb = np.load(f'./data/Baron_noresolution.npy')
imputeAdata = sc.AnnData(pd.DataFrame(imputeemb,index=refdf.index))
sc.pp.scale(imputeAdata)
sc.tl.pca(imputeAdata)
sc.pp.neighbors(imputeAdata)
sc.tl.umap(imputeAdata)

cluster_k_leiden(imputeAdata,numcls)
imputeAdata.obs['refleiden']=refAdata.obs['leiden']

In [ ]:
sc.pl.umap(imputeAdata,color='leiden')

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score,normalized_mutual_info_score,davies_bouldin_score,calinski_harabasz_score,silhouette_score

In [ ]:
normalized_mutual_info_score(imputeAdata.obs[f'refleiden'], imputeAdata.obs[f'cluster_{numcls}'])

In [ ]:
adjusted_rand_score(imputeAdata.obs[f'refleiden'], imputeAdata.obs[f'cluster_{numcls}'])

In [ ]:
silhouette_score(refAdata.obsm[f'X_pca'],imputeAdata.obs[f'cluster_{numcls}'])

### Zheng

In [ ]:
label = np.load('./data/zheng68k_label.npy',allow_pickle=True)

In [ ]:
imputeemb = np.load('./data/zheng68k_nores_nores_singlecell_cell_embedding_f1_resolution.npy')

numcls = np.unique(label).shape[0]

imputeAdata = sc.AnnData(pd.DataFrame(imputeemb))
sc.pp.scale(imputeAdata)
sc.tl.pca(imputeAdata)
sc.pp.neighbors(imputeAdata)
sc.tl.umap(imputeAdata)
imputeAdata.obs['label']=label.tolist()

In [ ]:
silhouette_score(imputeAdata.obsm['X_pca'],label)

In [ ]:
sc.pl.umap(imputeAdata,color='label')